سلول ۱ — شروع نوت‌بوک روز ۱۵

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(".")

GRAPH_DIR = PROJECT_ROOT / "Data_ml" / "graph_dataset"
EXPLAIN_DIR = GRAPH_DIR / "day13_explainability"
DAY14_DIR = GRAPH_DIR / "day14_case_studies"
DAY15_DIR = GRAPH_DIR / "day15_external_validation"

DAY15_DIR.mkdir(parents=True, exist_ok=True)

pred_df = pd.read_csv(EXPLAIN_DIR / "hgt_saved_fold_predictions.csv")

pairs_train = pd.read_csv(
    PROJECT_ROOT / "Data_proc" / "pairs" / "pairs_all_embedding_ready.csv"
)

print("pred_df:", pred_df.shape)
print("pairs_train:", pairs_train.shape)

display(pred_df.head())
display(pairs_train.head())

سلول ۲ — ساخت کلید تعاملات داخلی پروژه

In [ ]:
internal_pairs = set(
    pairs_train["pair_id"].astype(str)
)

print("internal pair count:", len(internal_pairs))

سلول ۳ — ساخت قالب فایل External Validation

فعلاً این فایل را می‌سازیم تا بعداً داده‌های UbiNet / ESIN / hUbiquitome / UniProt / UbiBrowser-new را داخلش بریزیم.

In [ ]:
external_template = pd.DataFrame(columns=[
    "source",
    "enzyme_class",
    "enz_gene",
    "sub_gene",
    "enz_ac",
    "sub_ac",
    "pair_id",
    "external_label",
    "evidence",
    "pmid"
])

external_template.to_csv(
    DAY15_DIR / "external_validation_template.csv",
    index=False
)

display(external_template)

سلول ۴ — پیدا کردن فایل‌های External

In [ ]:
from pathlib import Path

search_roots = [
    PROJECT_ROOT / "Data_raw",
    PROJECT_ROOT / "Data_interim",
    PROJECT_ROOT / "Data_proc",
]

patterns = [
    "*ubinet*",
    "*UbiNet*",
    "*ESIN*",
    "*esin*",
    "*huBi*",
    "*hUbi*",
    "*ubiquitome*",
    "*UniProt*",
    "*uniprot*",
    "*UbiBrowser*",
    "*ubibrowser*",
]

found_files = []

for root in search_roots:
    if not root.exists():
        continue
    
    for pattern in patterns:
        for f in root.rglob(pattern):
            if f.is_file():
                found_files.append(f)

found_files = sorted(set(found_files))

print("found files:", len(found_files))

for f in found_files:
    print(f)

سلول ۵ — دیدن ستون‌های فایل‌های پیدا شده

In [ ]:
import pandas as pd

for f in found_files:
    print("\n" + "="*100)
    print(f)
    
    try:
        if f.suffix.lower() == ".csv":
            df_tmp = pd.read_csv(f, nrows=5)
        elif f.suffix.lower() in [".tsv", ".txt"]:
            df_tmp = pd.read_csv(f, sep="\t", nrows=5)
        elif f.suffix.lower() in [".xlsx", ".xls"]:
            df_tmp = pd.read_excel(f, nrows=5)
        else:
            print("skipped unsupported:", f.suffix)
            continue
        
        print("shape preview:", df_tmp.shape)
        print("columns:", df_tmp.columns.tolist())
        display(df_tmp.head())
        
    except Exception as e:
        print("ERROR:", e)

مرحله بعد

ببین این فایل‌ها را داری یا نه:

UbiBrowser
UbiNet
ESIN
hUbiquitome
iUUCD

یا حتی:

ubiquitination
deubiquitination
E3 substrate
DUB substrate


In [ ]:
from pathlib import Path

keywords = [
    "ubi",
    "ubiquit",
    "e3",
    "dub",
    "substrate",
    "interaction",
    "esin",
    "ubinet",
    "iuucd",
]

hits = []

for root in [
    PROJECT_ROOT / "Data_raw",
    PROJECT_ROOT / "Data_interim",
    PROJECT_ROOT / "Data_proc",
]:
    
    if not root.exists():
        continue
    
    for f in root.rglob("*"):
        
        if not f.is_file():
            continue
        
        name = f.name.lower()
        
        if any(k in name for k in keywords):
            hits.append(str(f))

print("Found:", len(hits))

for h in sorted(hits):
    print(h)

سلول ۶ — بررسی فایل‌های UbiBrowser خام و قدیمی

In [ ]:
candidate_external_files = [
    PROJECT_ROOT / "Data_raw/UbiBrowser/Raw/E3-substrate-interactions.txt",
    PROJECT_ROOT / "Data_raw/UbiBrowser/Raw/DUB-substrate-interactions.txt",
    PROJECT_ROOT / "Data_raw/UbiBrowser/Modified_old/pos-E3-interactions.txt",
    PROJECT_ROOT / "Data_raw/UbiBrowser/Modified_old/pos-DUB-interactions.txt",
    PROJECT_ROOT / "Data_raw/UbiBrowser/Modified_old/uni_clean_E3_pos.csv",
    PROJECT_ROOT / "Data_raw/UbiBrowser/Modified_old/uni_clean_DUB_pos.csv",
    PROJECT_ROOT / "Data_interim/ubibrowser/e3_raw_read.csv",
    PROJECT_ROOT / "Data_interim/ubibrowser/dub_raw_read.csv",
    PROJECT_ROOT / "Data_interim/ubibrowser/e3_human_filtered.csv",
    PROJECT_ROOT / "Data_interim/ubibrowser/dub_human_filtered.csv",
]

for f in candidate_external_files:
    print("\n" + "="*100)
    print(f)
    print("exists:", f.exists())
    
    if not f.exists():
        continue
    
    try:
        if f.suffix.lower() == ".csv":
            df = pd.read_csv(f, nrows=5)
        else:
            try:
                df = pd.read_csv(f, sep="\t", nrows=5)
            except:
                df = pd.read_csv(f, sep=None, engine="python", nrows=5)
        
        print("columns:", df.columns.tolist())
        display(df.head())
        
    except Exception as e:
        print("ERROR:", e)

سلول ۷ — چک overlap خام‌ها با دیتاست داخلی

In [ ]:
def read_any_table(path):
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    
    for sep in ["\t", ",", ";"]:
        try:
            df = pd.read_csv(path, sep=sep)
            if df.shape[1] > 1:
                return df
        except:
            pass
    
    return pd.read_csv(path, sep=None, engine="python")


for f in candidate_external_files:
    if not f.exists():
        continue
    
    print("\n" + "="*100)
    print(f)
    
    df = read_any_table(f)
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())

In [ ]:
from pathlib import Path

for f in PROJECT_ROOT.rglob("*"):
    
    name = f.name.lower()
    
    if (
        "ubinet" in name
        or "esin" in name
        or "huibi" in name
        or "hubiquitome" in name
        or "iuucd" in name
    ):
        print(f)



Day 15 — مرحله ۱

دانلود UbiNet

اول باید UbiNet را بگیریم.

معمولاً UbiNet شامل:

* E3
* Substrate
* Interaction

است.

داخل پروژه بساز:

In [ ]:
DAY15_RAW = PROJECT_ROOT / "Data_raw" / "ExternalValidation"
DAY15_RAW.mkdir(parents=True, exist_ok=True)

In [ ]:
from pathlib import Path

import pandas as pd

UBINET_DIR = Path("./Data_raw/ExternalValidation/Ubinet")

for f in sorted(UBINET_DIR.glob("*")):

    print(f.name)

In [ ]:
for f in sorted(UBINET_DIR.glob("*")):
    print("\n" + "="*100)
    print(f.name)

    try:
        if f.suffix.lower() == ".csv":
            df = pd.read_csv(f, nrows=5)
        elif f.suffix.lower() in [".tsv", ".txt"]:
            df = pd.read_csv(f, sep="\t", nrows=5)
        elif f.suffix.lower() in [".xlsx", ".xls"]:
            df = pd.read_excel(f, nrows=5)
        else:
            print("unsupported")
            continue

        print(df.shape)
        print(df.columns.tolist())
        display(df.head())

    except Exception as e:
        print("ERROR:", e)

سلول ۱ — آماده‌سازی مسیرها و جدول کیس‌ها

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import time
import re

PROJECT_ROOT = Path(".")

GRAPH_DIR = PROJECT_ROOT / "Data_ml" / "graph_dataset"
DAY15_DIR = GRAPH_DIR / "day15_external_validation"
DAY15_DIR.mkdir(parents=True, exist_ok=True)

case_pairs = pd.DataFrame([
    {"enzyme": "MDM2",  "substrate": "TP53"},
    {"enzyme": "USP7",  "substrate": "TP53"},
    {"enzyme": "USP10", "substrate": "TP53"},
    {"enzyme": "STUB1", "substrate": "TP53"},
    {"enzyme": "CYLD",  "substrate": "TP53"},
    {"enzyme": "VHL",   "substrate": "HIF1A"},
    {"enzyme": "CBL",   "substrate": "EGFR"},
    {"enzyme": "BTRC",  "substrate": "CTNNB1"},
    {"enzyme": "ITCH",  "substrate": "NOTCH1"},
    {"enzyme": "ITCH",  "substrate": "DVL2"},
])

display(case_pairs)

سلول ۲ — گرفتن accession از فایل‌های خود پروژه

In [ ]:
pairs_train = pd.read_csv(
    PROJECT_ROOT / "Data_proc" / "pairs" / "pairs_all_embedding_ready.csv",
    dtype=str,
    low_memory=False
)

gene_to_ac = {}

for _, r in pairs_train.iterrows():
    if pd.notna(r.get("enz_gene")) and pd.notna(r.get("enz_ac")):
        gene_to_ac[str(r["enz_gene"])] = str(r["enz_ac"])
    if pd.notna(r.get("sub_gene")) and pd.notna(r.get("sub_ac")):
        gene_to_ac[str(r["sub_gene"])] = str(r["sub_ac"])

case_pairs["enzyme_ac"] = case_pairs["enzyme"].map(gene_to_ac)
case_pairs["substrate_ac"] = case_pairs["substrate"].map(gene_to_ac)

display(case_pairs)

print("missing enzyme accession:")
display(case_pairs[case_pairs["enzyme_ac"].isna()])

print("missing substrate accession:")
display(case_pairs[case_pairs["substrate_ac"].isna()])

سلول ۳ — دانلود JSON از UniProt برای هر accession

In [ ]:
def fetch_uniprot_json(accession, sleep=0.2):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    r = requests.get(url, timeout=30)
    time.sleep(sleep)
    
    if r.status_code != 200:
        return None, r.status_code, r.text[:300]
    
    return r.json(), r.status_code, ""

all_accessions = sorted(
    set(case_pairs["enzyme_ac"].dropna()) |
    set(case_pairs["substrate_ac"].dropna())
)

uniprot_records = {}
fetch_log = []

for acc in all_accessions:
    js, status, err = fetch_uniprot_json(acc)
    uniprot_records[acc] = js
    
    fetch_log.append({
        "accession": acc,
        "status": status,
        "error": err
    })
    
    print(acc, status)

fetch_log = pd.DataFrame(fetch_log)
display(fetch_log)

fetch_log.to_csv(
    DAY15_DIR / "uniprot_fetch_log.csv",
    index=False
)

سلول ۴ — استخراج evidence از commentهای UniProt

In [ ]:
def flatten_text(obj):
    texts = []
    
    if obj is None:
        return ""
    
    if isinstance(obj, str):
        return obj
    
    if isinstance(obj, dict):
        for v in obj.values():
            texts.append(flatten_text(v))
    
    elif isinstance(obj, list):
        for v in obj:
            texts.append(flatten_text(v))
    
    return " ".join([t for t in texts if t])


def get_uniprot_comment_text(record):
    if record is None:
        return ""
    
    comments = record.get("comments", [])
    return flatten_text(comments)


def get_uniprot_xrefs_text(record):
    if record is None:
        return ""
    
    xrefs = record.get("uniProtKBCrossReferences", [])
    return flatten_text(xrefs)


def get_pmids_from_record(record):
    txt = flatten_text(record)
    pmids = sorted(set(re.findall(r"\b\d{7,9}\b", txt)))
    return ";".join(pmids)


for acc, rec in uniprot_records.items():
    print("=" * 80)
    print(acc)
    print(get_uniprot_comment_text(rec)[:1000])

سلول ۵ — اعتبارسنجی اینکه اسم interaction در UniProt entry آمده یا نه

In [ ]:
validation_rows = []

for _, r in case_pairs.iterrows():
    enz = r["enzyme"]
    sub = r["substrate"]
    enz_ac = r["enzyme_ac"]
    sub_ac = r["substrate_ac"]
    
    enz_record = uniprot_records.get(enz_ac)
    sub_record = uniprot_records.get(sub_ac)
    
    enz_text = (
        get_uniprot_comment_text(enz_record)
        + " "
        + get_uniprot_xrefs_text(enz_record)
    ).upper()
    
    sub_text = (
        get_uniprot_comment_text(sub_record)
        + " "
        + get_uniprot_xrefs_text(sub_record)
    ).upper()
    
    enz_mentions_sub = sub.upper() in enz_text or str(sub_ac).upper() in enz_text
    sub_mentions_enz = enz.upper() in sub_text or str(enz_ac).upper() in sub_text
    
    validation_rows.append({
        "interaction": f"{enz} -> {sub}",
        "enzyme": enz,
        "substrate": sub,
        "enzyme_ac": enz_ac,
        "substrate_ac": sub_ac,
        "uniprot_enzyme_mentions_substrate": enz_mentions_sub,
        "uniprot_substrate_mentions_enzyme": sub_mentions_enz,
        "uniprot_validated": enz_mentions_sub or sub_mentions_enz,
        "enzyme_pmids": get_pmids_from_record(enz_record),
        "substrate_pmids": get_pmids_from_record(sub_record),
    })

uniprot_validation = pd.DataFrame(validation_rows)

display(uniprot_validation)

uniprot_validation.to_csv(
    DAY15_DIR / "uniprot_curated_validation_cases.csv",
    index=False
)

سلول ۶ — خلاصه اعتبارسنجی UniProt

In [ ]:
print("n cases:", len(uniprot_validation))
print("UniProt validated:", int(uniprot_validation["uniprot_validated"].sum()))
print("Validation rate:", uniprot_validation["uniprot_validated"].mean() * 100)

display(
    uniprot_validation[
        [
            "interaction",
            "uniprot_validated",
            "uniprot_enzyme_mentions_substrate",
            "uniprot_substrate_mentions_enzyme",
            "enzyme_pmids",
            "substrate_pmids"
        ]
    ]
)

باید بررسی کنیم آیا دو مورد شکست خورده واقعاً interaction شناخته‌شده هستند یا نه.

In [ ]:
display(
    uniprot_validation[
        ~uniprot_validation["uniprot_validated"]
    ]
)

In [ ]:
for acc in [
    "Q9NQC7",   # CYLD
    "Q96J02",   # ITCH
]:
    rec = uniprot_records[acc]

    txt = (
        get_uniprot_comment_text(rec)
        + " "
        + get_uniprot_xrefs_text(rec)
    )

    print("="*120)
    print(acc)
    print(txt[:5000])

آیا interactionهایی که مدل با confidence بالا پیش‌بینی می‌کند واقعاً از نظر زیستی معنادارند؟

بیاییم Top100 interaction مدل را استخراج کنیم.

In [ ]:
top100 = (
    pred_df
    .sort_values(
        "prob_hgt_saved",
        ascending=False
    )
    .head(100)
    .copy()
)

print(top100.shape)

display(
    top100[
        [
            "enz_gene",
            "sub_gene",
            "prob_hgt_saved",
            "y_true"
        ]
    ].head(20)
)

ببینیم چند interaction مربوط به مسیرهای معروف سرطان هستند.

In [ ]:
cancer_genes = {
    "TP53",
    "EGFR",
    "CTNNB1",
    "MYC",
    "AKT1",
    "YAP1",
    "NOTCH1",
    "HIF1A",
    "CCND1",
    "RIPK1",
    "RIPK2",
    "TRAF2",
    "TRAF6",
    "SQSTM1"
}

top100["contains_cancer_gene"] = (
    top100["enz_gene"].isin(cancer_genes)
    |
    top100["sub_gene"].isin(cancer_genes)
)

top100["contains_cancer_gene"].mean() * 100

خروجی را مرتب کنیم.

In [ ]:
cancer_hits = top100[
    top100["contains_cancer_gene"]
].copy()

display(
    cancer_hits[
        [
            "enz_gene",
            "sub_gene",
            "prob_hgt_saved"
        ]
    ]
)

فراوانی substrateها.

In [ ]:
substrate_freq = (
    top100["sub_gene"]
    .value_counts()
    .reset_index()
)

substrate_freq.columns = [
    "substrate",
    "count"
]

display(substrate_freq.head(30))

فراوانی E3ها.

In [ ]:
e3_freq = (
    top100[
        top100["pair_id"].str.startswith("E3")
    ]["enz_gene"]
    .value_counts()
    .reset_index()
)

e3_freq.columns = [
    "enzyme",
    "count"
]

display(e3_freq.head(20))

فراوانی DUBها.

In [ ]:
dub_freq = (
    top100[
        top100["pair_id"].str.startswith("DUB")
    ]["enz_gene"]
    .value_counts()
    .reset_index()
)

dub_freq.columns = [
    "enzyme",
    "count"
]

display(dub_freq.head(20))